In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pyproj
import os
import warnings

warnings.filterwarnings("ignore")

class LocalCFVisualizer:
    def __init__(self):
        self.input_dir = "../data/processed"
        self.output_dir = "../data/outputs"
        os.makedirs(self.output_dir, exist_ok=True)
        
        self.scenarios = ['hist', 'ssp126', 'ssp245', 'ssp370', 'ssp585']
        self.seasons = ['DJF', 'MAM', 'JJA', 'SON']
        self.extent = [94, 142, -12, 9]

        self.clon, self.clat = 120.0, -2.5
        proj_std = pyproj.Proj(f"+proj=merc +lon_0={self.clon} +lat_ts={self.clat}")
        _, y_center = proj_std(self.clon, self.clat)
        self.data_crs = ccrs.Mercator(central_longitude=self.clon, latitude_true_scale=self.clat, false_northing=-y_center)

        # Distinct color limits (vmin/vmax) for absolute vs delta projections
        self.config = {
            'CF_SOLAR': {
                'name': 'Solar PV', 
                'vmin_abs': 15, 'vmax_abs': 55, 'cmap_abs': 'OrRd', 'extend_abs': 'both',
                'vmin_delta_hist': -6, 'vmax_delta_hist': 6, 
                'vmin_delta_ssp245': -3, 'vmax_delta_ssp245': 3,  
                'cmap_delta': 'RdBu_r'
            },
            'CF_SOLAR_siang': {
                'name': 'Solar PV (Daylight)', 
                'vmin_abs': 35, 'vmax_abs': 75, 'cmap_abs': 'OrRd', 'extend_abs': 'both',
                'vmin_delta_hist': -6, 'vmax_delta_hist': 6, 
                'vmin_delta_ssp245': -3, 'vmax_delta_ssp245': 3, 
                'cmap_delta': 'RdBu_r'
            },
            'CF_WIND': {
                'name': 'Wind Power', 
                'vmin_abs': 0, 'vmax_abs': 70, 'cmap_abs': 'YlGnBu', 'extend_abs': 'max',
                'vmin_delta_hist': -15, 'vmax_delta_hist': 15, 
                'vmin_delta_ssp245': -8, 'vmax_delta_ssp245': 8, 
                'cmap_delta': 'RdBu_r'
            }
        }

    def load_local_nc(self, prefix, var_prefix, scen):
        fpath = os.path.join(self.input_dir, f"ENS_{prefix}_{var_prefix}_{scen}.nc")
        if not os.path.exists(fpath): return None
        with xr.open_dataset(fpath) as ds: return ds[list(ds.data_vars)[0]].load()

    def prep_map(self, ax):
        ax.set_extent(self.extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.7, edgecolor='black', zorder=2)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='dimgray', linestyle='--', zorder=2)
        ax.spines['geo'].set_linewidth(0.5)

    def add_label_box(self, ax, text, loc='bottom_left'):
        kwargs = dict(transform=ax.transAxes, fontsize=14, fontweight='bold', 
                      bbox=dict(boxstyle='square,pad=0.2', facecolor='white', alpha=0.85, edgecolor='gray', zorder=5))
        if loc == 'bottom_right':
            ax.text(0.96, 0.04, text, ha='right', va='bottom', **kwargs)
        elif loc == 'bottom_left':
            ax.text(0.04, 0.04, text, ha='left', va='bottom', **kwargs)

    def format_seasonal_layout(self, ax, i, j, season, top_label):
        if j == 0:
            ax.text(-0.06, 0.5, season, va='center', ha='center', rotation=90, transform=ax.transAxes, fontsize=18, fontweight='bold')
        if i == 0 and top_label:
            ax.set_title(top_label, fontsize=18, fontweight='bold', pad=8)

    def _add_continuous_colorbar(self, fig, cfg, ax_rect, plot_type='absolute'):
        if plot_type == 'absolute':
            vmin, vmax, cmap, extend_cbar = cfg['vmin_abs'], cfg['vmax_abs'], cfg['cmap_abs'], cfg['extend_abs']
            prefix = ""
        elif plot_type == 'delta_hist':
            vmin, vmax, cmap, extend_cbar = cfg['vmin_delta_hist'], cfg['vmax_delta_hist'], cfg['cmap_delta'], 'both'
            prefix = "$\\Delta$ "
        else:
            vmin, vmax, cmap, extend_cbar = cfg['vmin_delta_ssp245'], cfg['vmax_delta_ssp245'], cfg['cmap_delta'], 'both'
            prefix = "$\\Delta$ "
            
        cbar_ax = fig.add_axes(ax_rect)
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
        cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', extend=extend_cbar)
        cbar.set_label(f"{prefix}CF: {cfg['name']} [%]", fontsize=20, fontweight='bold')
        cbar.ax.tick_params(labelsize=20)

    def plot_seasonal_absolute(self, var_key):
        print(f"   [INFO] Generating Seasonal Absolute plot for {var_key}...")
        fig, axes = plt.subplots(4, 5, figsize=(20, 8), subplot_kw={'projection': ccrs.PlateCarree()})
        cfg = self.config[var_key]
        for i, season in enumerate(self.seasons):
            for j, scen in enumerate(self.scenarios):
                ax = axes[i, j]
                self.prep_map(ax)
                da = self.load_local_nc('Season', var_key, scen)
                if da is not None:
                    (da.sel(season=season) * 100).plot.imshow(x='lon', y='lat', ax=ax, transform=self.data_crs, 
                                                              vmin=cfg['vmin_abs'], vmax=cfg['vmax_abs'], cmap=cfg['cmap_abs'], 
                                                              add_colorbar=False, interpolation='bilinear', zorder=1, add_labels=False)
                self.format_seasonal_layout(ax, i, j, season, scen.upper())

        plt.subplots_adjust(wspace=0.01, hspace=-0.05, bottom=0.08, top=0.92, left=0.05, right=0.98)
        self._add_continuous_colorbar(fig, cfg, [0.05, 0.04, 0.93, 0.03], plot_type='absolute')
        plt.savefig(os.path.join(self.output_dir, f"GRID_Seasonal_Absolute_{var_key}.png"), dpi=300, bbox_inches='tight')
        plt.show() 
        plt.close()

    def plot_seasonal_delta(self, var_key, baseline='hist'):
        print(f"   [INFO] Generating Seasonal Delta plot (vs {baseline.upper()}) for {var_key}...")
        
        plot_type = 'delta_hist' if baseline == 'hist' else 'delta_ssp245'
        
        if baseline == 'hist':
            scens = self.scenarios[1:] 
            cols = 4
            figsize = (16, 8)
            title_suffix = ""
        else:
            scens = ['ssp126', 'ssp370', 'ssp585'] 
            cols = 3
            figsize = (12, 8) 
            title_suffix = " vs 245"
            
        fig, axes = plt.subplots(4, cols, figsize=figsize, subplot_kw={'projection': ccrs.PlateCarree()})
        cfg = self.config[var_key]
        da_base = self.load_local_nc('Season', var_key, baseline)
        
        vmin = cfg[f'vmin_{plot_type}']
        vmax = cfg[f'vmax_{plot_type}']
        
        for i, season in enumerate(self.seasons):
            for j, scen in enumerate(scens):
                ax = axes[i, j]
                self.prep_map(ax)
                da_fut = self.load_local_nc('Season', var_key, scen)
                
                if da_fut is not None and da_base is not None:
                    ((da_fut.sel(season=season) - da_base.sel(season=season)) * 100).plot.imshow(
                                                                      x='lon', y='lat', ax=ax, transform=self.data_crs, 
                                                                      vmin=vmin, vmax=vmax, cmap=cfg['cmap_delta'], 
                                                                      add_colorbar=False, interpolation='bilinear', zorder=1, add_labels=False)
                
                self.format_seasonal_layout(ax, i, j, season, f"$\\Delta$ {scen.upper()}{title_suffix}")

        plt.subplots_adjust(wspace=0.01, hspace=-0.19, bottom=0.08, top=0.95, left=0.05, right=0.98)
        self._add_continuous_colorbar(fig, cfg, [0.05, 0.05, 0.93, 0.03], plot_type=plot_type)
        plt.savefig(os.path.join(self.output_dir, f"GRID_Seasonal_Delta_vs_{baseline}_{var_key}.png"), dpi=300, bbox_inches='tight')
        plt.show() 
        plt.close()

    def plot_climatology(self, var_key, plot_type='absolute'):
        print(f"   [INFO] Generating Climatology plot ({plot_type}) for {var_key}...")
        
        is_delta = 'delta' in plot_type
        
        if plot_type == 'absolute':
            rows, cols = 3, 2
            scens = self.scenarios
            fig_height = 8
            baseline = None
            vmin, vmax, cmap = self.config[var_key]['vmin_abs'], self.config[var_key]['vmax_abs'], self.config[var_key]['cmap_abs']
        elif plot_type == 'delta_hist':
            rows, cols = 2, 2
            scens = self.scenarios[1:]
            fig_height = 5
            baseline = 'hist'
            vmin, vmax, cmap = self.config[var_key]['vmin_delta_hist'], self.config[var_key]['vmax_delta_hist'], self.config[var_key]['cmap_delta']
        else:
            rows, cols = 2, 2
            scens = ['ssp126', 'ssp370', 'ssp585']
            fig_height = 5
            baseline = 'ssp245'
            vmin, vmax, cmap = self.config[var_key]['vmin_delta_ssp245'], self.config[var_key]['vmax_delta_ssp245'], self.config[var_key]['cmap_delta']
            
        fig, axes = plt.subplots(rows, cols, figsize=(10, fig_height), subplot_kw={'projection': ccrs.PlateCarree()})
        axes_flat = axes.flatten()
        cfg = self.config[var_key]
        
        da_base = self.load_local_nc('Clim', var_key, baseline) if is_delta else None
        
        for i, scen in enumerate(scens):
            ax = axes_flat[i]
            self.prep_map(ax)
            da = self.load_local_nc('Clim', var_key, scen)
            
            if da is not None:
                if is_delta and da_base is not None:
                    data = (da - da_base) * 100
                else:
                    data = da * 100
                    
                data.plot.imshow(x='lon', y='lat', ax=ax, transform=self.data_crs, 
                                 vmin=vmin, vmax=vmax, cmap=cmap, 
                                 add_colorbar=False, interpolation='bilinear', zorder=1, add_labels=False)
            
            label_text = f"$\\Delta$ {scen.upper()} vs 245" if plot_type == 'delta_ssp245' else (f"$\\Delta$ {scen.upper()}" if is_delta else scen.upper())
            self.add_label_box(ax, label_text, loc='bottom_left')
            
        for j in range(len(scens), len(axes_flat)):
            axes_flat[j].remove()

        if is_delta:
            plt.subplots_adjust(wspace=-0.035, hspace=0.02, bottom=0.15, top=0.95, left=0.05, right=0.98)
            self._add_continuous_colorbar(fig, cfg, [0.05, 0.075, 0.93, 0.04], plot_type=plot_type)
        else:
            plt.subplots_adjust(wspace=0.01, hspace=-0.275, bottom=0.08, top=0.95, left=0.05, right=0.98)
            self._add_continuous_colorbar(fig, cfg, [0.05, 0.075, 0.93, 0.03], plot_type=plot_type)

        plt.savefig(os.path.join(self.output_dir, f"GRID_Climatology_{plot_type}_{var_key}.png"), dpi=300, bbox_inches='tight')
        plt.show() 
        plt.close()
    
    def run_pipeline(self):
        print("[INFO] INITIALIZING CAPACITY FACTOR VISUALIZATION PIPELINE...")
        for var_key in self.config.keys():
            print(f"\n{'='*50}\nProcessing: {var_key}\n{'='*50}")
            
            self.plot_seasonal_absolute(var_key)
            self.plot_climatology(var_key, plot_type='absolute')
            
            self.plot_seasonal_delta(var_key, baseline='hist')
            self.plot_climatology(var_key, plot_type='delta_hist')
            
            self.plot_seasonal_delta(var_key, baseline='ssp245')
            self.plot_climatology(var_key, plot_type='delta_ssp245')
            
        print("\n[SUCCESS] All plotting modes completed successfully!")

if __name__ == "__main__":
    LocalCFVisualizer().run_pipeline()

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pyproj
import os
import warnings

warnings.filterwarnings("ignore")

# DIRECTORY CONFIGURATION
in_dir = "../data/processed"
out_dir = "../data/outputs"
os.makedirs(out_dir, exist_ok=True)

ssps = ['ssp126', 'ssp245', 'ssp370', 'ssp585']
seasons = ['DJF', 'MAM', 'JJA', 'SON']
extent = [94, 142, -12, 9]

# --- MERCATOR PARAMETERS FROM REGCM NAMELIST ---
clon, clat = 120.0, -2.5
proj_std = pyproj.Proj(f"+proj=merc +lon_0={clon} +lat_ts={clat}")
_, y_center = proj_std(clon, clat)
data_crs = ccrs.Mercator(central_longitude=clon, latitude_true_scale=clat, false_northing=-y_center)

def prep_map(ax):
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7, edgecolor='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='dimgray', linestyle='--', zorder=3)

def add_label_box(ax, text, loc='bottom_left'):
    kwargs = dict(transform=ax.transAxes, fontsize=12, fontweight='normal', 
                  bbox=dict(boxstyle='square,pad=0.2', facecolor='white', alpha=0.85, edgecolor='gray', zorder=5))
    if loc == 'bottom_right':
        ax.text(0.96, 0.04, text, ha='right', va='bottom', **kwargs)
    elif loc == 'bottom_left':
        ax.text(0.04, 0.04, text, ha='left', va='bottom', **kwargs)

def format_seasonal_layout(ax, i, j, season, top_label):
    if j == 0:
        ax.text(-0.06, 0.5, season, va='center', ha='center', rotation=90, transform=ax.transAxes, fontsize=18, fontweight='bold')
    if i == 0 and top_label:
        ax.set_title(top_label, fontsize=18, fontweight='bold', pad=8)

def _add_continuous_colorbar(fig, img, alpha, ax_rect):
    cbar_ax = fig.add_axes(ax_rect)
    cbar = fig.colorbar(img, cax=cbar_ax, orientation='horizontal', extend='both')
    cbar.set_label(f"Δ CF [% per decade] (Hatched area = Insignificant, p ≥ {alpha})", fontsize=20, fontweight='bold')
    cbar.ax.tick_params(labelsize=20)
    
def plot_trend_annual(var_type):
    # Configure 2x2 matrix layout
    fig, axes = plt.subplots(2, 2, figsize=(10, 5), subplot_kw={'projection': ccrs.PlateCarree()})
    axes = axes.flatten()
    
    cmap = 'RdBu_r' 
    
    # Dynamic detection for Solar variables
    vmax = 1.5 if 'SOLAR' in var_type else 5
    vmin = -vmax
    
    alpha = 0.05
    scale_factor = 10
    
    img = None
    for i, ssp in enumerate(ssps):
        ax = axes[i]
        prep_map(ax)
        
        fpath = os.path.join(in_dir, f"TREND_60YR_{var_type}_Ensemble_{ssp}.nc")
        if not os.path.exists(fpath): continue
        
        with xr.open_dataset(fpath) as ds:
            slope = ds['slope_annual'] * scale_factor * 100
            pval = ds['pval_annual'].fillna(1.0)
            lon, lat = ds.lon, ds.lat
            
            img = ax.pcolormesh(lon, lat, slope, transform=data_crs, 
                                cmap=cmap, vmin=vmin, vmax=vmax, shading='auto', zorder=1)
            
            ax.contourf(lon, lat, pval, levels=[alpha, np.inf], transform=data_crs,
                        colors='none', hatches=['...'], zorder=2)
            
        # Position label at bottom left
        add_label_box(ax, f"Trend {ssp.upper()}", loc='bottom_left')
            
    # Layout adjustment matching Climatology Delta 2x2
    plt.subplots_adjust(wspace=-0.035, hspace=0.02, bottom=0.15, top=0.95, left=0.05, right=0.98)
    if img: _add_continuous_colorbar(fig, img, alpha, [0.05, 0.075, 0.93, 0.04])
    
    out_fpath = os.path.join(out_dir, f"SPATIAL_TREND_60YR_ANNUAL_{var_type}.jpg")
    plt.savefig(out_fpath, dpi=300, bbox_inches='tight')
    plt.show() 
    plt.close()
    print(f"[SAVED] {os.path.basename(out_fpath)}")

def plot_trend_seasonal(var_type):
    # Configure 4x4 matrix layout
    fig, axes = plt.subplots(4, 4, figsize=(16, 8), subplot_kw={'projection': ccrs.PlateCarree()})
    
    cmap = 'RdBu_r' 
    
    # Dynamic detection for Solar variables
    vmax = 1.5 if 'SOLAR' in var_type else 5
    vmin = -vmax
    
    alpha = 0.05
    scale_factor = 10
    
    img = None
    for i, season in enumerate(seasons):
        for j, ssp in enumerate(ssps):
            ax = axes[i, j]
            prep_map(ax)
            
            fpath = os.path.join(in_dir, f"TREND_60YR_{var_type}_Ensemble_{ssp}.nc")
            if not os.path.exists(fpath): continue
            
            with xr.open_dataset(fpath) as ds:
                slope = ds[f'slope_{season}'] * scale_factor * 100
                pval = ds[f'pval_{season}'].fillna(1.0)
                lon, lat = ds.lon, ds.lat
                
                img = ax.pcolormesh(lon, lat, slope, transform=data_crs, 
                                    cmap=cmap, vmin=vmin, vmax=vmax, shading='auto', zorder=1)
                
                ax.contourf(lon, lat, pval, levels=[alpha, np.inf], transform=data_crs,
                            colors='none', hatches=['...'], zorder=2)
            
            format_seasonal_layout(ax, i, j, season, f"Trend {ssp.upper()}")
            
    # Layout adjustment matching Seasonal Delta 4x4
    plt.subplots_adjust(wspace=0.01, hspace=-0.19, bottom=0.08, top=0.95, left=0.05, right=0.98)
    if img: _add_continuous_colorbar(fig, img, alpha, [0.05, 0.05, 0.93, 0.03])
    
    out_fpath = os.path.join(out_dir, f"SPATIAL_TREND_60YR_SEASONAL_{var_type}.jpg")
    plt.savefig(out_fpath, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"[SAVED] {os.path.basename(out_fpath)}")

# Run visualization pipeline
for vt in ['CF_SOLAR', 'CF_SOLAR_siang', 'CF_WIND']:
    plot_trend_annual(vt)
    plot_trend_seasonal(vt)